In [ ]:

from typing import Any, Dict, List, Optional

from pydantic import BaseModel

from agente_avaliacao_imagens.schemas import AnaliseImagens, FeedbackImagens


class ReActInput(BaseModel):
    """Entrada do agente ReAct de análise de imagens."""

    fotos_urls: List[str] = []
    api_key: Optional[str] = None

c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from phoenix.otel import register

tracer_provider = register(
  project_name="agente-react-imoveis",
  auto_instrument=True
)

08/05/2026 01:31:42 PM 📋 Ensuring phoenix working directory: C:\Users\jefer\.phoenix
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.schemas
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.tables
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.types
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.constraints
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.defaults
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.comments


OpenTelemetry Tracing Details
|  Phoenix Project: agente-react-imoveis
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\phoenix\otel\otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


In [3]:
import logging
from typing import List

from langchain_core.tools import tool

from agente_avaliacao_imagens.prompts import PROMPT_DESCREVER_FOTO
from agente_avaliacao_imagens.utils import processar_todos_lotes

logger = logging.getLogger(__name__)


@tool
async def descrever_fotos(fotos_urls: List[str]) -> str:
    """Processa as fotos do imóvel e devolve a descrição técnica de cada imagem.

    Use esta ferramenta para obter a descrição das fotos. Depois, com base nela,
    preencha a análise estruturada final (scores, problemas, pontos fortes).

    Args:
        fotos_urls: lista de URLs das fotos do imóvel.
    """
    if not fotos_urls:
        return "Nenhuma URL de foto fornecida."
    
    logger.info(f"Processando {len(fotos_urls)} fotos para descrição.")

    descricao = await processar_todos_lotes(fotos_urls, 5, prompt=PROMPT_DESCREVER_FOTO)
    if not descricao:
        logger.error("Nao foi possivel descrever as fotos.")
        return "Falha ao descrever as fotos."
    return descricao

In [14]:
import json
import logging
import os
from typing import List, Optional

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.prebuilt import create_react_agent

from agente_avaliacao_imagens.schemas import AnaliseImagens
#from .tools import descrever_fotos

logger = logging.getLogger(__name__)

MODELO_AGENTE = os.getenv("MODELO_AGENTE_IMAGENS",
                          #"z-ai/glm-5.2"#V
                          "meta/llama-3.1-8b-instruct"V
                          #"nvidia/nemotron-3-ultra-550b-a55b" #V
                          # "google/gemma-4-31b-it" X
                          #"poolside/laguna-xs-2.1" #X
                          #"qwen/qwen-2.5-72b-instruct" #X
                          ) #"moonshotai/kimi-k2.6")"deepseek-ai/deepseek-v4-flash")

SYSTEM_PROMPT = """Você é um engenheiro civil e especialista em avaliação de imóveis para house flipping.

Sua tarefa é analisar as fotos de um imóvel e produzir um relatório técnico estruturado.

Passos:
1. Chame a ferramenta `descrever_fotos` com as URLs das fotos. Ela devolverá a descrição técnica de cada imagem.
2. Analise a descrição recebida e preencha a análise estruturada final com os campos abaixo.

Definição de cada campo do relatório final:

- **score_conservacao (float 0-10):** condição geral de conservação/mainutenção do que está visível (infiltrações, trincas, desgaste, estado de paredes/teto).
- **score_acabamento (float 0-10):** qualidade/padrão dos materiais (piso, revestimentos, metais, portas, esquadrias).
- **score_potencial_reforma (float 0-10):** o quanto é viável/vantajoso reformar o espaço (nota alta = boa estrutura que valoriza com melhorias; nota baixa = exige demolição pesada ou já está em ótimo estado).
- **confianca_imagem (float 0-10):** o quanto a descrição é confiável, clara e útil para uma avaliação técnica.
- **imagem_aceitavel (bool):** `true` se a foto mostra elementos reais do imóvel e é clara; `false` se for irrelevante (selfie, parede escura, objeto aleatório) ou a descrição for vaga demais.
- **problemas_visiveis (List[str]):** patologias, defeitos, danos ou sinais de desgaste identificados. Vazio se não houver.
- **pontos_fortes (List[str]):** aspectos positivos observados (iluminação natural, piso em bom estado, acabamento moderno, área espaçosa). Vazio se não houver.
- **observacoes (str):** resumo da opinião técnica. Se `imagem_aceitavel = false` ou as notas forem baixas, use este campo para justificar tecnicamente.

Regras importantes:
- Baseie-se APENAS na descrição fornecida pela ferramenta. Nunca invente ou infira o que não está visível.
- Se um aspecto não puder ser avaliado, pondere as notas de forma neutra e registre a limitação em `observacoes`.
- Se as fotos não retratarem um ambiente de imóvel (imagem irrelevante/ilegível), marque `imagem_aceitavel = false`, atribua `0.0` a todos os scores e explique em `observacoes`.
- Responda SEMPRE em português.
"""


def criar_agente_imagens(api_key: Optional[str] = None):
    model = ChatNVIDIA(
        model=MODELO_AGENTE,
        api_key=api_key or os.getenv("NVIDIA_API_KEY"),
    )
    return create_react_agent(
        model=model,
        tools=[descrever_fotos],
        prompt=SYSTEM_PROMPT,
        response_format=AnaliseImagens,
    )


async def analisar_imagens(
    fotos_urls: List[str],
    api_key: Optional[str] = None,
) -> AnaliseImagens:
    """Executa o agente ReAct de análise de imagens e devolve o relatório estruturado."""
    agente = criar_agente_imagens(api_key=api_key)

    mensagem_usuario = {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Analise as fotos do imóvel:\n"
                    + json.dumps(fotos_urls, ensure_ascii=False, indent=2)
                ),
            }
        ]
    }

    resultado = await agente.ainvoke(mensagem_usuario)
    resposta = resultado.get("structured_response")

    if isinstance(resposta, dict):
        return AnaliseImagens(**resposta)
    return resposta

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1975982450.py, line 16)

In [2]:
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

In [ ]:
fotos = df['fotos'].iloc[6].tolist()

In [ ]:
fotos

['https://resizedimgs.zapimoveis.com.br/img/vr-listing/2be5cbec55da625c7063443f287d852b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/80f6aae4cdef3bb2baf1c8609934408b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/532e18b9c870fd26cfc2a712304896ab/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/ed7f7a3d250cc07765b1d97fa3e94448/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/fa37d92847ba04c856f07049f497b1c3/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zap

In [ ]:
response

AnaliseImagens(score_conservacao=8.0, score_acabamento=8.0, score_potencial_reforma=5.0, confianca_imagem=8.5, imagem_aceitavel=True, problemas_visiveis=[], pontos_fortes=[' Fachada de edifício', 'Bom estado de conservação', 'Design contemporâneo'], observacoes='A imagem é redundante, repetindo a fachada exibida nas fotos 1 e 5[20, 39]. Analisa apenas o exterior do edifício, que não apresenta danos visíveis[37]. Ausência de informações sobre os ambientes internos do imóvel[41].')

In [2]:
import pandas as pd
import json

In [40]:
#df = pd.read_json('olx_alugueis.json', lines=True)
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df_09 = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-09.parquet')
df_08 = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

#with open(PASTA_DADOS/ 'joinville_aluguel_olx_2026-08.json', 'r', encoding='utf-8') as f:
#df = json.load(f)

# Verificando o urls que existiam no mes anterior

In [41]:
import pandas as pd
from collections import Counter

df_04 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-04.parquet')
df_05 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-05.parquet')
df_06 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-06.parquet')
df_07 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-07.parquet')
df_08 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-08.parquet')
df_09 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-09.parquet')

# --- Tamanho por mes ---
print('=== Registros por mes ===')
for i, df in enumerate([df_04, df_05, df_06, df_07, df_08, df_09], 4):
    print(f'  {i:02d}/2026: {len(df):>6} registros, {df.shape[1]} colunas')

# --- URLs por mes ---
urls = {}
for m, df in [(4, df_04), (5, df_05), (6, df_06), (7, df_07), (8, df_08), (9, df_09)]:
    urls[m] = set(df['url'].dropna())

# --- Intersecao com setembro ---
print('\n=== Intersecao com setembro (09) ===')
for m in [4, 5, 6, 7, 8]:
    inter = urls[m] & urls[9]
    so_antigo = urls[m] - urls[9]
    so_09 = urls[9] - urls[m]
    print(f'  {m:02d} vs 09: ambos={len(inter)}, so_no_{m:02d}={len(so_antigo)}, so_no_09={len(so_09)}')

# --- Novos e removidos por transicao ---
print('\n=== Novos por mes (nao existiam no mes anterior) ===')
for m in [5, 6, 7, 8, 9]:
    novos = urls[m] - urls[m - 1]
    print(f'  Novos em {m:02d}: {len(novos)}')

print('\n=== Removidos por mes (existiam no anterior) ===')
for m in [5, 6, 7, 8, 9]:
    removidos = urls[m - 1] - urls[m]
    print(f'  Removidos do {m - 1:02d} para {m:02d}: {len(removidos)}')

# --- Persistencia ---
todas_urls = []
for m in [4, 5, 6, 7, 8, 9]:
    todas_urls.extend(urls[m])

contagem = Counter(todas_urls)
persistencia = Counter(contagem.values())

print('\n=== Persistencia (em quantos meses cada URL aparece) ===')
for meses, qtd in sorted(persistencia.items()):
    print(f'  {meses} meses: {qtd} URLs')



=== Registros por mes ===
  04/2026:  34573 registros, 33 colunas
  05/2026:  26905 registros, 33 colunas
  06/2026:  24336 registros, 62 colunas
  07/2026:  23152 registros, 62 colunas
  08/2026:  25312 registros, 32 colunas
  09/2026:  21725 registros, 38 colunas

=== Intersecao com setembro (09) ===
  04 vs 09: ambos=6754, so_no_04=27819, so_no_09=14971
  05 vs 09: ambos=8123, so_no_05=18782, so_no_09=13602
  06 vs 09: ambos=7199, so_no_06=17137, so_no_09=14526
  07 vs 09: ambos=10154, so_no_07=12998, so_no_09=11571
  08 vs 09: ambos=11595, so_no_08=13717, so_no_09=10130

=== Novos por mes (nao existiam no mes anterior) ===
  Novos em 05: 6201
  Novos em 06: 12506
  Novos em 07: 8275
  Novos em 08: 11697
  Novos em 09: 10130

=== Removidos por mes (existiam no anterior) ===
  Removidos do 04 para 05: 13869
  Removidos do 05 para 06: 15075
  Removidos do 06 para 07: 9459
  Removidos do 07 para 08: 9537
  Removidos do 08 para 09: 13717

=== Persistencia (em quantos meses cada URL apar

In [44]:
urls_comuns = urls[4] & urls[5] & urls[6] & urls[7] & urls[8] & urls[9]

df_filtrado = df_09[df_09['url'].isin(urls_comuns)]

print(f'URLs presentes em todos os 6 meses: {len(urls_comuns)}')
print(f'Registros em df_09 filtrados: {len(df_filtrado)}')

URLs presentes em todos os 6 meses: 4093
Registros em df_09 filtrados: 4093


In [56]:
df_filtrado[df_filtrado['tipo_imovel'].isin(['apartamento'])]['url'].to_dict()

{1: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-3-quartos-com-garagem-sc-joinville-anita-garibaldi-248m2-RS1399000/id-38997600/',
 4: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-3-quartos-com-garagem-sc-joinville-santo-antonio-RS2311250/id-32728703/',
 17: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-3-quartos-com-garagem-sc-joinville-santo-antonio-135m2-RS1490000/id-32728478/',
 24: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-3-quartos-com-garagem-sc-joinville-america-322m2-RS1050000/id-35991604/',
 32: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-4-quartos-com-garagem-sc-joinville-america-322m2-RS950000/id-36821331/',
 35: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-3-quartos-com-garagem-sc-joinville-atiradores-199m2-RS2850000/id-34704127/',
 48: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-3-quartos-com-garagem-sc-joinville-america-213m2-RS2198000/id-28360929/',
 53: 'https://www.ch

In [54]:
df_filtrado.groupby(['bairro', 'tipo_imovel'])['preco_por_m2'].aggregate(['mean', 'median','count']).head(50)

mean        median  count
bairro              tipo_imovel                                   
adhemar garcia      apartamento  4.759932e+03   4545.248869      8
                    casa         4.379798e+03   4195.312500     15
america             apartamento  8.731313e+03   8512.570245    438
                    casa         7.937946e+03   8022.001901     80
                    comercial    9.889333e+03   9842.519685      7
                    outros       1.028657e+04  10443.812500      8
                    terreno      3.283631e+03   1499.571551      3
anita garibaldi     apartamento  1.175905e+04   8138.333333    339
                    casa         4.015038e+04   6800.000000     83
                    outros       1.129011e+04  11290.109659      2
area rural          rural        2.331007e+01     23.310071      1
atiradores          apartamento           inf   8896.368602    168
                    casa         6.501956e+03   7009.933775     22
                    comercial    1.263238e+04  12632.376712      2
                    outros       9.233068e+03   9044.115385      7
                    terreno      8.333333e+03   8333.333333      1
aventureiro         apartamento  5.245354e+03   5357.142857     17
                    casa         4.548441e+03   5075.757576     47
boa vista           apartamento  4.682296e+03   5000.000000      9
                    casa         6.409173e+03   5541.666667     57
                    comercial    4.590164e+03   4590.163934      1
                    outros       5.428571e+03   5428.571429      1
boehmerwald         apartamento  5.206580e+03   5127.104377      6
                    casa         4.192422e+03   4125.000000     21
bom retiro          apartamento  7.071495e+03   6989.065934     80
                    casa         5.900259e+03   5860.447761    108
                    comercial    1.853282e+03   1853.281853      1
bucarein            apartamento  7.079029e+03   6869.827586     49
                    casa         5.593917e+03   6181.818182     21
                    outros       7.812500e+03   7812.500000      1
centro              apartamento  8.470279e+03   8291.666667    138
                    casa         5.254862e+03   4305.555556      6
                    comercial    6.393259e+03   5539.617486     22
                    outros       1.564780e+04  15647.799593      2
comasa              apartamento  4.432059e+03   4245.901639      5
                    casa         4.858102e+03   5000.000000     25
costa e silva       apartamento           inf   6515.218750    224
                    casa         1.050257e+04   5620.512821    146
                    comercial    6.381573e+03   6381.573098      2
                    outros       7.511934e+03   7431.081081      5
                    terreno      1.302083e+03   1302.083333      1
distrito industrial apartamento  6.200993e+03   6200.992949      2
                    casa         6.265238e+03   6265.238095      2
espinheiros         apartamento  5.922293e+03   5060.200000      5
                    casa         4.637436e+03   4848.450057     22
                    terreno      1.624103e+03   1650.000000      3
fatima              apartamento           inf   5816.062802     10
                    casa         2.654187e+03   2395.273632      8
                    terreno      4.333333e+03   4333.333333      1
floresta            apartamento  5.786069e+03   5541.530769     50

In [51]:
import pandas as pd
import numpy as np

# --- Filtrar URLs presentes em todos os 6 meses ---
urls_comuns = urls[4] & urls[5] & urls[6] & urls[7] & urls[8] & urls[9]
df_filtrado = df_09[df_09['url'].isin(urls_comuns)].copy()

print(f'URLs presentes em todos os 6 meses: {len(urls_comuns)}')
print(f'Registros em df_09 filtrados: {len(df_filtrado)}')
print()

# --- 1. Visao geral por tipo ---
print('=== Distribuicao por tipo de imovel ===')
tipo_resumo = df_filtrado.groupby('tipo_imovel').agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
    metragem_media=('metragem', 'mean'),
    valor_medio=('valor_imovel', 'mean'),
).round(0)
print(tipo_resumo.to_string())
print()

# --- 2. Top bairros por preco m2 ---
print('=== Top 15 bairros por preco m2 medio ===')
bairro_resumo = df_filtrado.groupby('bairro').agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
    metragem_media=('metragem', 'mean'),
    valor_medio=('valor_imovel', 'mean'),
).round(0)
bairro_resumo = bairro_resumo[bairro_resumo['qtd'] >= 5].sort_values('preco_m2_medio', ascending=False)
print(bairro_resumo.head(15).to_string())
print()

# --- 3. Cruzamento bairro x tipo (preco m2 medio) ---
print('=== Preco m2 medio: bairro x tipo (top 30) ===')
cruzado = df_filtrado.groupby(['bairro', 'tipo_imovel']).agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
).round(0)
cruzado = cruzado[cruzado['qtd'] >= 3].sort_values('preco_m2_medio', ascending=False)
print(cruzado.head(30).to_string())
print()

# --- 4. Faixas de preco por bairro ---
print('=== Distribuicao por faixa de preco por bairro (top 10 bairros) ===')
top_bairros = df_filtrado['bairro'].value_counts().head(10).index
faixa_bairro = df_filtrado[df_filtrado['bairro'].isin(top_bairros)].groupby(['bairro', 'faixa']).size().unstack(fill_value=0)
print(faixa_bairro.to_string())
print()

# --- 5. Metricas por quartos ---
print('=== Preco m2 por quantidade de quartos ===')
quartos_resumo = df_filtrado.groupby('quartos').agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
    metragem_media=('metragem', 'mean'),
    valor_medio=('valor_imovel', 'mean'),
).round(0)
print(quartos_resumo.to_string())
print()

# --- 6. Fonte dos dados ---
print('=== Distribuicao por fonte ===')
print(df_filtrado['fonte'].value_counts().to_string())

URLs presentes em todos os 6 meses: 4093
Registros em df_09 filtrados: 4093

=== Distribuicao por tipo de imovel ===
              qtd  preco_m2_medio  preco_m2_mediana  metragem_media  valor_medio
tipo_imovel                                                                     
apartamento  2222             inf            7294.0           200.0     970248.0
casa         1751             inf            5645.0           317.0     877629.0
comercial      37          7152.0            6667.0           122.0     781029.0
galpao          4          2035.0            2262.0           574.0     851250.0
outros         39          9496.0            8865.0           122.0    1265178.0
rural          12          2213.0              79.0         37084.0    2640833.0
terreno        28          2354.0            1456.0          7662.0    1268227.0

=== Top 15 bairros por preco m2 medio ===
                 qtd  preco_m2_medio  preco_m2_mediana  metragem_media  valor_medio
bairro                     